# Colab runner — Experiment 03
Clone/sync, install, execute independent stages, and push aggregate results. Use GPU if available; L4 preferred. A fresh output directory is required to avoid overwriting checkpoints.

In [ ]:
from pathlib import Path
import subprocess, os
repo=Path('/content/pragmatiq-scalable-explainable')
if not repo.exists():
    subprocess.run(['git','clone','-b','research','https://github.com/haider4445/pragmatiq-scalable-explainable.git',str(repo)],check=True)
os.chdir(repo)
subprocess.run(['git','switch','research'],check=True)
subprocess.run(['git','pull','--ff-only','origin','research'],check=True)

In [ ]:
%pip install -e '.[train]'

In [ ]:
import os, subprocess, sys
os.environ['PRAGMATIQ_DISABLE_FLASH']='1'
subprocess.run([sys.executable,'research/experiments/03_pipeline_decomposition.py'],check=True)

In [ ]:
import json, pandas as pd
from pathlib import Path
r=json.loads(Path('research/results/03_pipeline_metrics.json').read_text())
t=pd.Series(r['stage_seconds'],name='seconds')
display(t.to_frame().assign(share_pct=100*t/t.sum()))
t.plot.barh(title='Pipeline stage time')
print(r['probe'])

## Commit results
Review aggregate metrics. Never upload model weights or raw customer records. If GitHub CLI is missing, install it before this cell.

In [ ]:
from google.colab import userdata
import os, subprocess, shutil
assert shutil.which('gh'), 'Install gh first: !apt-get install -y gh'
env=dict(os.environ, GH_TOKEN=userdata.get('GITHUB_TOKEN'))
subprocess.run(['gh','auth','setup-git'],env=env,check=True)
subprocess.run(['git','config','user.name','haider4445'],check=True)
if subprocess.run(['git','config','user.email'],capture_output=True).returncode != 0:
    subprocess.run(['git','config','user.email','34925624+haider4445@users.noreply.github.com'],check=True)
files=['research/results/03_pipeline_metrics.json']
subprocess.run(['git','add','--',*files],check=True)
subprocess.run(['git','diff','--cached','--stat'],check=True)

In [ ]:
subprocess.run(['git','commit','-m','experiment: record independent pipeline stage metrics','--',*files],check=True)
subprocess.run(['git','push','origin','research'],env=env,check=True)
del env
subprocess.run(['git','log','-1','--oneline'],check=True)